# 06 - Machine Learning with scikit-learn 🤖

We built k-NN and linear regression **from scratch** (files `01`–`05`), so we know exactly what's inside. Now let's use **scikit-learn**, the standard ML library, which does the same things in a few lines and offers dozens more models.

Every scikit-learn model follows the same pattern:
```python
model = SomeModel(settings)         # 1. choose a model
model.fit(X_train, y_train)         # 2. learn from training data
predictions = model.predict(X_test) # 3. predict on new data
model.score(X_test, y_test)         # 4. evaluate
```
- **`X`** = the features (a table: one row per example, one column per feature)
- **`y`** = the target (the answer we want to predict)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

penguins = pd.read_csv("data/penguins.csv").dropna()
penguins.head()

## Part 1: Classification. Which species is this penguin?

In [ ]:
FEATURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
X = penguins[FEATURES]
y = penguins["species"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)   # stratify = keep species proportions the same
print(len(X_train), "training penguins,", len(X_test), "test penguins")

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
print("Accuracy (raw features):", round(knn.score(X_test, y_test), 3))

### Scaling matters (just like in `04_train_test_accuracy.py`)
A **pipeline** chains steps: first standardize the features, then run k-NN. The scaler learns the mean and std from the **training** data only.

In [ ]:
scaled_knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
scaled_knn.fit(X_train, y_train)
print("Accuracy (scaled):", round(scaled_knn.score(X_test, y_test), 3))

### Where does it make mistakes? The confusion matrix
Rows are the **true** species and columns are the **predicted** species. Everything off the diagonal is a mistake.

In [ ]:
predictions = scaled_knn.predict(X_test)
pd.crosstab(y_test, predictions, rownames=["true"], colnames=["predicted"])

### Choosing k: cross-validation
If we pick k by checking the **test** set over and over, we're quietly "studying for the exam with the answer key." **Cross-validation** instead splits the *training* data into 5 parts and rotates which part is held out.

In [ ]:
ks = range(1, 31)
cv_scores = []
for k in ks:
    model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    cv_scores.append(cross_val_score(model, X_train, y_train, cv=5).mean())

plt.plot(ks, cv_scores, marker="o")
plt.xlabel("k (number of neighbors)")
plt.ylabel("Cross-validated accuracy")
plt.title("Choosing k")
plt.grid(alpha=0.3)
plt.show()
best_k = ks[int(np.argmax(cv_scores))]
print("Best k:", best_k)

### Predicting a brand-new penguin

In [ ]:
new_penguin = pd.DataFrame([[45.0, 15.0, 215.0, 5000.0]], columns=FEATURES)
print(scaled_knn.predict(new_penguin))
for species, share in zip(scaled_knn.classes_, scaled_knn.predict_proba(new_penguin)[0]):
    print(f"  {species}: {share:.0%} of the neighbors")

### A different model: the decision tree 🌳
A decision tree learns a flowchart of yes/no questions. It's easy to read, and you can see *why* it decided.

In [ ]:
tree = DecisionTreeClassifier(max_depth=3, random_state=0)
tree.fit(X_train, y_train)
print("Tree accuracy:", round(tree.score(X_test, y_test), 3))
print(export_text(tree, feature_names=FEATURES))

## Part 2: Regression. How heavy is this penguin?

In [ ]:
X1 = penguins[["flipper_length_mm"]]          # double brackets: X must be a TABLE (2D)
y1 = penguins["body_mass_g"]
X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.2, random_state=42)

line = LinearRegression()
line.fit(X1_train, y1_train)
print(f"mass = {line.coef_[0]:.1f} * flipper + ({line.intercept_:.1f})")    # compare with 05_linear_regression.py!
print("R^2 on test data:", round(line.score(X1_test, y1_test), 3))

### Multiple regression: use more features
Can we do better with bill size and sex too? Models need numbers, so we turn `sex` into a 0/1 column.

In [ ]:
data = penguins.copy()
data["is_male"] = (data["sex"] == "MALE").astype(int)
MORE = ["flipper_length_mm", "bill_length_mm", "bill_depth_mm", "is_male"]

X2_train, X2_test, y2_train, y2_test = train_test_split(data[MORE], data["body_mass_g"], test_size=0.2, random_state=42)
multi = LinearRegression().fit(X2_train, y2_train)
print("R^2 with 4 features:", round(multi.score(X2_test, y2_test), 3))
for name, coef in zip(MORE, multi.coef_):
    print(f"  {name:<18} {coef:+8.1f}")

In [ ]:
pred = multi.predict(X2_test)
plt.scatter(y2_test, pred, alpha=0.6)
plt.plot([2500, 6500], [2500, 6500], "r--", label="perfect prediction")
plt.xlabel("True mass (g)")
plt.ylabel("Predicted mass (g)")
plt.legend()
plt.title("Predicted vs actual")
plt.show()
print(f"Typical error: {np.mean(np.abs(pred - y2_test)):.0f} g")

## Summary

| Task | Predicts | Models we used | Metric |
|------|----------|----------------|--------|
| **Classification** | a category | `KNeighborsClassifier`, `DecisionTreeClassifier` | accuracy, confusion matrix |
| **Regression** | a number | `LinearRegression` | R², average error |

**The ML workflow:**
1. Frame the question
2. Get and clean the data
3. Split into train and test
4. Prepare the features (e.g. scaling)
5. Fit
6. Evaluate **on the test set**
7. Compare with a simple **baseline**
8. Improve

## ✏️ Try it
1. Use only `bill_length_mm` and `bill_depth_mm` for the species classifier. How much accuracy do you lose?
2. Try `DecisionTreeClassifier(max_depth=1)` and `max_depth=10`. What happens to test accuracy?
3. Add `island` to the regression (hint: `pd.get_dummies(data["island"])`). Does R² improve?

In [ ]:
# your code here